In [ ]:
# 📌 INSTALAR LIBRERÍAS NECESARIAS
!pip install gspread pandas scikit-learn oauth2client

import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 📌 1️⃣ CONECTAR CON GOOGLE SHEETS
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
credenciales = ServiceAccountCredentials.from_json_keyfile_name("CREDENCIALES.json", scope)
cliente = gspread.authorize(credenciales)

# 📌 2️⃣ ABRIR GOOGLE SHEETS
spreadsheet_id = "1UVxV--6Zn-KyxLpJs986_xd-CuJBXC3n9kjN6m1npg0"  # ID de la hoja
hoja = cliente.open_by_key(spreadsheet_id).sheet1  # Primera hoja

# 📌 3️⃣ LEER LOS DATOS DESDE GOOGLE SHEETS
datos = pd.DataFrame(hoja.get_all_records())

# 📌 4️⃣ PREPROCESAR LOS DATOS Y CONVERTIRLOS A NÚMEROS
columnas_numericas = ["Humedad del sensor", "ARENA (%)", "ARCILLA (%)", "Humedad al tacto (%)"]

for col in columnas_numericas:
    datos[col] = datos[col].replace("", np.nan)  # Reemplazar valores vacíos con NaN
    datos[col] = pd.to_numeric(datos[col], errors="coerce")  # Convertir a número

# 📌 5️⃣ MANEJO DE VARIABLES CATEGÓRICAS (Compactación y Pedregosidad)
categorical_columns = ["Compactación", "Pedregosidad"]

for col in categorical_columns:
    datos[col] = datos[col].astype(str).fillna("Media")  # Reemplazar NaN con "Media"

# Convertir categorías en variables numéricas (One-Hot Encoding)
datos = pd.get_dummies(datos, columns=categorical_columns, drop_first=True)

# 📌 6️⃣ DETERMINAR EL TIPO DE SUELO SEGÚN PROPORCIÓN ARENA/ARCILLA
datos["Proporción Arena/Arcilla"] = datos["ARENA (%)"] / datos["ARCILLA (%)"]
datos["Tipo de Suelo"] = datos["Proporción Arena/Arcilla"].apply(
    lambda x: "Arenoso" if x > 4 else ("Franco" if 2.5 <= x <= 4 else "Arcilloso")
)

# 📌 7️⃣ ENTRENAR MODELOS RANDOM FOREST PARA CADA TIPO DE SUELO
def entrenar_modelo(tipo_suelo):
    df = datos[datos["Tipo de Suelo"] == tipo_suelo]
    if df.empty:
        return None, None, None, None

    # Selección de características
    features = ["Humedad del sensor", "ARENA (%)", "ARCILLA (%)"] + list(df.filter(like="Compactación_").columns) + list(df.filter(like="Pedregosidad_").columns)

    X_train = df[features]
    y_train = df["Humedad al tacto (%)"]

    # Eliminar filas con valores nulos en y_train
    X_train = X_train[~y_train.isna()]
    y_train = y_train.dropna()

    if len(y_train) < 5:  # Evitar entrenar modelos con pocos datos
        return None, None, None, features

    modelo = RandomForestRegressor(n_estimators=100, random_state=42)
    modelo.fit(X_train, y_train)

    # 📌 Calcular R² y RMSE
    y_pred = modelo.predict(X_train)
    r2 = r2_score(y_train, y_pred)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))

    return modelo, r2, rmse, features  # Se devuelve también la lista de columnas usadas

modelo_arenoso, r2_arenoso, rmse_arenoso, features_arenoso = entrenar_modelo("Arenoso")
modelo_franco, r2_franco, rmse_franco, features_franco = entrenar_modelo("Franco")
modelo_arcilloso, r2_arcilloso, rmse_arcilloso, features_arcilloso = entrenar_modelo("Arcilloso")

# 📌 8️⃣ MOSTRAR EL AJUSTE DEL MODELO (R²)
print("\n📊 **Evaluación del Modelo Random Forest con Compactación y Pedregosidad**")
print(f"🌱 **Suelos Arenosos** → R²: {r2_arenoso:.4f}, RMSE: {rmse_arenoso:.2f}")
print(f"🌿 **Suelos Francos** → R²: {r2_franco:.4f}, RMSE: {rmse_franco:.2f}")
print(f"🌾 **Suelos Arcillosos** → R²: {r2_arcilloso:.4f}, RMSE: {rmse_arcilloso:.2f}")

# 📌 9️⃣ PREDECIR HUMEDAD AL TACTO PARA TODAS LAS FILAS (TENGAN O NO DATO REAL)
def predecir_humedad(fila):
    if fila["Tipo de Suelo"] == "Arenoso" and modelo_arenoso:
        X_test = pd.DataFrame([fila[features_arenoso].values], columns=features_arenoso)
        return modelo_arenoso.predict(X_test)[0]
    elif fila["Tipo de Suelo"] == "Franco" and modelo_franco:
        X_test = pd.DataFrame([fila[features_franco].values], columns=features_franco)
        return modelo_franco.predict(X_test)[0]
    elif fila["Tipo de Suelo"] == "Arcilloso" and modelo_arcilloso:
        X_test = pd.DataFrame([fila[features_arcilloso].values], columns=features_arcilloso)
        return modelo_arcilloso.predict(X_test)[0]
    else:
        return None  # Si no hay modelo, dejar vacío

# Aplicar predicción a TODAS las filas (independiente de si tienen o no "Humedad al tacto (%)")
datos["Humedad al tacto estimada (%)"] = datos.apply(predecir_humedad, axis=1)

# 📌 🔟 ACTUALIZAR SOLO LA COLUMNA "Humedad al tacto estimada (%)" EN GOOGLE SHEETS
hoja.update(f"G2:G{len(datos) + 1}", datos[["Humedad al tacto estimada (%)"]].values.tolist())

print("\n✅ Datos actualizados en Google Sheets con Compactación y Pedregosidad.")

# 📌 1️⃣ FUNCIÓN PARA GENERAR FÓRMULA EN GOOGLE SHEETS
def generar_formula_google_sheets(importancias, features):
    """
    Convierte los coeficientes del modelo en una fórmula de Google Sheets.
    """
    formula = "=("  # Comenzamos la fórmula

    for coef, feature in zip(importancias, features):
        columna = feature.replace(" ", "_").upper()  # Convertir nombres de variables a formato de columna
        formula += f"{coef:.6f} * INDIRECT(\"{columna}\"), "

    formula = formula.rstrip(", ") + ")"  # Eliminar la última coma y cerrar la fórmula
    return formula

# 📌 2️⃣ GENERAR FÓRMULAS PARA CADA TIPO DE SUELO
formulas = {}
if modelo_arenoso:
    formulas["Arenoso"] = generar_formula_google_sheets(modelo_arenoso.feature_importances_, features_arenoso)
if modelo_franco:
    formulas["Franco"] = generar_formula_google_sheets(modelo_franco.feature_importances_, features_franco)
if modelo_arcilloso:
    formulas["Arcilloso"] = generar_formula_google_sheets(modelo_arcilloso.feature_importances_, features_arcilloso)

# 📌 3️⃣ GUARDAR FÓRMULAS EN GOOGLE SHEETS
hoja.update("J1", [["Tipo de Suelo", "Fórmula Google Sheets"]])
hoja.update("J2", [[tipo, formula] for tipo, formula in formulas.items()])

print("✅ Fórmulas generadas y guardadas en Google Sheets.")

/tmp/ipython-input-3410793950.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  datos[col] = datos[col].replace("", np.nan)  # Reemplazar valores vacíos con NaN



📊 **Evaluación del Modelo Random Forest con Compactación y Pedregosidad**
🌱 **Suelos Arenosos** → R²: 0.8931, RMSE: 4.20
🌿 **Suelos Francos** → R²: 0.8736, RMSE: 4.07
🌾 **Suelos Arcillosos** → R²: 0.8982, RMSE: 3.35


/tmp/ipython-input-3410793950.py:102: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  hoja.update(f"G2:G{len(datos) + 1}", datos[["Humedad al tacto estimada (%)"]].values.tolist())



✅ Datos actualizados en Google Sheets con Compactación y Pedregosidad.


/tmp/ipython-input-3410793950.py:130: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  hoja.update("J1", [["Tipo de Suelo", "Fórmula Google Sheets"]])
/tmp/ipython-input-3410793950.py:131: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  hoja.update("J2", [[tipo, formula] for tipo, formula in formulas.items()])


✅ Fórmulas generadas y guardadas en Google Sheets.
